## Crime Data Preprocessing Notebook

## Overview
This notebook loads, inspects and cleans UK crime data across 4 police forces 
Metropolitan, West Yorkshire, Devon & Cornwall, North Yorkshire covering 2024-01 to 2025-08.

The cleaned data will be saved as CSV files for use in the analysis notebook.

## Contents
1. Street Data - Loading, Inspection & Cleaning
2. Stop & Search Data - Loading, Inspection & Cleaning
3. ONS Population Data - Merge & Export

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

SECTION 1 Loading one CSV file first to inspect the structure, understand the columns 
and identify any data quality issues before doing the same for the full dataset.

In [ ]:
#Loading a single file to inspect
df = pd.read_csv('2024-01/2024-01-metropolitan-street.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.tail()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

##Checking the main collumns which need to be analysed to understand the different crime types, police forces and time periods.

In [ ]:
# Distribution of crime types
df["Crime type"].value_counts()

In [ ]:
# Police forces in the dataset
df["Falls within"].value_counts()

In [ ]:
# Months covered
df["Month"].value_counts()

Based on the previous information cleaning decsions were made. These being:
Collumn data was going to be removed as no analysis can be done with it. 
Drop rows were longitude and latitude are null as they cannot be used for mapping.
Delete any duplicated rows.
Convert the month to a valid datetime as its currently stored as String (seen in df.info)

In [ ]:
# Keep original safe, work on a copy
df_clean = df.copy()

In [ ]:
#Scatter plot to see all points are linked to graph.
plt.scatter(df_clean['Longitude'], df_clean['Latitude'])
plt.show()

In [58]:
# Drop Context column - 100% empty
df_clean.drop(columns=['Context'], inplace=True)
# Drop rows with missing location data
df_clean.dropna(subset=['Longitude', 'Latitude'], inplace=True)
# Drop duplicate rows
df_clean.drop_duplicates(inplace=True)
# Convert Month to datetime
df_clean['Month'] = pd.to_datetime(df_clean['Month'])

In [ ]:
print(df_clean.isnull().sum())
print(df_clean.shape)

The cleaning process has been cleaned on one file, we can use the GLOB library to clean all the street CSV files. 

In [61]:
# Load all street CSVs across all forces and months
all_files = glob.glob('**/*-street.csv', recursive=True)
print(f"Total files found: {len(all_files)}")

Total files found: 96


In [62]:
# Combine all files into one dataframe
df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
df.shape

(3313260, 12)

In [ ]:
df.info()
df.isnull().sum()
df.duplicated().sum()

In [69]:
df_clean = df.copy()

In [ ]:
#Copy the exact same cleaning methods as we did with the single file. 
df_clean.drop(columns=['Context'], inplace=True)
df_clean.dropna(subset=['Longitude', 'Latitude'], inplace=True)
df_clean.drop_duplicates(inplace=True)
df_clean['Month'] = pd.to_datetime(df_clean['Month'])

In [ ]:
df_clean.isnull().sum()
df_clean.shape

In [90]:
# Save clean street data
df_clean.to_csv('clean_street.csv', index=False)
print("Street data saved successfully!")

KeyboardInterrupt: 

In [74]:
# Test clean street data loaded correctly
test = pd.read_csv('clean_street.csv')
test.shape

(3039777, 11)

Ran a test to see if all the coordinates are between the 4 police forces which will be used for analysis. 

In [ ]:
plt.scatter(test['Longitude'], test['Latitude'])
plt.show()

Two points appeared outside the main cluster with Longitude below -5. 
These were investigated and found to belong to Devon & Cornwall Police, specifically Cornwall and the Isles of Scilly. 

In [ ]:
test[test['Longitude'] < -5]

Running final validation checks on the clean dataset to confirm the data is ready for analysis.

In [ ]:
test['Crime type'].value_counts()
print(f"Total unique crime types: {test['Crime type'].nunique()}")

In [ ]:
print(test['Month'].value_counts().sort_index())
print(f"\nTotal months: {test['Month'].nunique()}")

In [91]:
print(test['Falls within'].value_counts())
print(f"\nTotal forces: {test['Falls within'].nunique()}")

Falls within
Metropolitan Police Service    2062333
West Yorkshire Police           591387
Devon & Cornwall Police         272161
North Yorkshire Police          113896
Name: count, dtype: int64

Total forces: 4


Section 3: Loading the ONS population estimates for 2024 for further analysis. 

In [ ]:
#Loading the population data
ons = pd.read_csv('police_population_2024.csv')
ons.head()

In [ ]:
#Keeping only relevant columns for analysis
ons_clean = ons[['Police_Force', 'Population_2024']]
ons_clean

,Police_Force,Population_2024
0,Metropolitan Police Service,"9,074,625"
1,North Yorkshire Police,"844,571"
2,West Yorkshire Police,"2,435,236"
3,Devon & Cornwall Police,"1,840,161"


In [98]:
#Combine population data with crime data
ons_clean = ons_clean.rename(columns={'Police_Force': 'Falls within'})
df_final = test.merge(ons_clean, on='Falls within')
df_final.shape

(3039777, 12)

In [ ]:
#Test to see if population data was added to the final dataframe.. 
df_final.head()